In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer - Une fonction Qiskit par Kipu Quantum

> **Note:** * Les fonctions Qiskit sont une fonctionnalité expérimentale disponible uniquement pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et sont susceptibles d'évoluer.

## Vue d'ensemble
Avec l'Iskay Quantum Optimizer de Kipu Quantum, tu peux aborder des problèmes d'optimisation complexes à l'aide des ordinateurs quantiques IBM&reg;. Ce solveur s'appuie sur l'algorithme de pointe [bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) de Kipu, qui ne nécessite que la fonction objectif en entrée pour livrer automatiquement les solutions aux problèmes. Il peut traiter des problèmes d'optimisation impliquant jusqu'à 156 qubits, ce qui permet d'utiliser tous les qubits des appareils quantiques IBM. L'Optimizer utilise un mappage 1-pour-1 entre les variables classiques et les qubits, ce qui te permet de traiter des problèmes d'optimisation comportant jusqu'à 156 variables binaires.

L'Optimizer permet de résoudre des problèmes d'optimisation binaire non contraints. En plus de la formulation QUBO (Quadratic Unconstrained Binary Optimization) couramment utilisée, il prend également en charge les problèmes d'optimisation d'ordre supérieur (HUBO). Le solveur utilise un algorithme quantique non variationnel, effectuant la majeure partie du calcul sur des appareils quantiques.

Ce qui suit fournit plus de détails sur l'algorithme utilisé et un guide succinct sur l'utilisation de la fonction, ainsi que des résultats de benchmark sur diverses instances de problèmes de tailles et complexités différentes.
## Description
L'Optimizer est une implémentation clé en main d'algorithmes d'optimisation quantique de pointe. Il résout des problèmes d'optimisation en exécutant des circuits quantiques hautement compressés sur du matériel quantique. Cette compression est obtenue en introduisant des termes contradiabatiques dans l'évolution temporelle sous-jacente du système quantique. L'algorithme effectue plusieurs itérations d'exécutions matérielles pour obtenir les solutions finales et les combine avec un post-traitement. Ces étapes sont intégrées de manière transparente dans le flux de travail de l'Optimizer et sont exécutées automatiquement.

### Comment fonctionne le Quantum Optimizer ?
Cette section présente les bases de l'algorithme bf-DCQO implémenté. Une introduction à l'algorithme est également disponible sur la [chaîne YouTube de Qiskit.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

L'algorithme repose sur l'évolution temporelle d'un système quantique qui se transforme au fil du temps, la solution du problème étant encodée dans l'état fondamental du système quantique à la fin de l'évolution. D'après le [théorème adiabatique](https://en.wikipedia.org/wiki/Adiabatic_theorem), cette évolution doit être lente pour garantir que le système reste dans son état fondamental. La numérisation de cette évolution est la base du calcul adiabatique quantique numérisé (DQA) et du célèbre algorithme QAOA. Cependant, l'évolution lente requise n'est pas réalisable pour des tailles de problèmes croissantes, car elle entraîne une profondeur de circuit croissante. En utilisant des protocoles contradiabatiques, on peut supprimer les excitations indésirables se produisant pendant de courtes durées d'évolution tout en restant dans l'état fondamental. Ici, la numérisation de cette durée d'évolution plus courte produit des circuits quantiques de moindre profondeur avec moins de portes enchevêtrantes.

Les circuits des algorithmes bf-DCQO utilisent généralement jusqu'à dix fois moins de portes enchevêtrantes que DQA, et trois à quatre fois moins de portes enchevêtrantes que les implémentations QAOA standard. En raison du plus faible nombre de portes, moins d'erreurs surviennent lors de l'exécution des circuits sur le matériel. Ainsi, l'optimizer n'a pas besoin de recourir à des techniques de suppression ou d'atténuation des erreurs. Leur implémentation dans les versions futures pourra améliorer encore davantage la qualité des solutions.

Bien que l'algorithme bf-DCQO utilise des itérations, il est non variationnel. Après chaque itération de l'algorithme, la distribution des états est mesurée. La distribution obtenue est utilisée pour calculer un soi-disant champ de biais. Ce champ de biais permet de démarrer l'itération suivante à partir d'un état d'énergie proche de la solution précédemment trouvée. Ainsi, l'algorithme converge à chaque itération vers des solutions d'énergie plus basse. En règle générale, une dizaine d'itérations suffisent pour converger vers une solution, nécessitant au total un nombre d'itérations bien inférieur à celui des algorithmes variationnels, qui est de l'ordre d'une centaine d'itérations.

L'optimizer combine l'algorithme bf-DCQO avec un post-traitement classique. Après avoir mesuré la distribution des états, une recherche locale est effectuée. Lors de la recherche locale, les bits de la solution mesurée sont retournés aléatoirement. Après le retournement, l'énergie du nouveau bitstring est évaluée. Si l'énergie est plus basse, le bitstring est conservé comme nouvelle solution. La recherche locale ne croît que linéairement avec le nombre de qubits ; elle est donc peu coûteuse en termes de calcul. Comme le post-traitement corrige les retournements de bits locaux, il compense les erreurs de flip de bit qui résultent souvent des imperfections matérielles et des erreurs de lecture.

### Flux de travail
Voici un schéma du flux de travail du Quantum Optimizer.

![Workflow](../docs/images/guides/kipu-optimization/workflow.svg "Flux de travail du Quantum Optimizer")

En utilisant le Quantum Optimizer, la résolution d'un problème d'optimisation sur du matériel quantique peut être réduite à :
* Formuler la fonction objectif du problème
* Accéder à l'Optimizer via les fonctions Qiskit
* Exécuter l'Optimizer et collecter le résultat
## Benchmarks
Les métriques de benchmark ci-dessous montrent que l'Optimizer traite efficacement des problèmes impliquant jusqu'à 156 qubits et offrent un aperçu général de la précision et de la scalabilité de l'optimizer sur différents types de problèmes. Note que les métriques de performance réelles peuvent varier en fonction des caractéristiques spécifiques du problème, telles que le nombre de variables, la densité et la localité des termes dans la fonction objectif, et l'ordre polynomial.

Le tableau suivant inclut le ratio d'approximation (AR), une métrique définie comme suit :
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
où $C$ est la fonction objectif, $C_{\textrm{min}}$, $C_{\textrm{max}}$ sont ses valeurs minimale et maximale, et $C^{*}$ est le coût de la meilleure solution trouvée, respectivement. Ainsi, AR=100\% signifie que l'état fondamental du problème a été obtenu.

| Exemple           | Nombre de qubits | Ratio d'approximation | Temps total (s) | Utilisation runtime (s) | Nombre total de shots | Nombre d'itérations |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| MaxCut non pondéré |        28        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut non pondéré |        30        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut non pondéré |        32        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut non pondéré |        80        |   100%   |      480       |        60         |          90k          |          9           |
| MaxCut non pondéré |       100        |   100%   |      330       |        60         |          60k          |          6           |
| MaxCut non pondéré |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- Les instances MaxCut avec 28, 30 et 32 qubits ont été exécutées sur ibm_sherbrooke. Les instances avec 80, 100 et 120 ont été exécutées sur un processeur Heron r2.
- Les instances HUBO ont également été exécutées sur un processeur Heron r2.

Toutes les instances de benchmark sont accessibles sur GitHub (voir [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Un exemple pour exécuter ces instances se trouve dans [Exemple 3 : Instances de benchmark](#example-3-benchmark-instances).
## Entrées et sorties
### Entrée
Consulte le tableau suivant pour tous les paramètres d'entrée acceptés par le Quantum Optimizer. La section **Options** qui suit fournit plus de détails sur les `options` disponibles.
| Nom         | Type               | Description                                                                                                                                                                            | Requis | Défaut                                                                                       | Exemple                                                                                                              |
| ------------ | ------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------- | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Les coefficients du problème d'optimisation formulé en format QUBO/HUBO ou spin. Pour plus d'informations sur la spécification du problème, voir [Formats de problème acceptés](#accepted-problem-formats)                                       | Oui      | N/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | Indique si les coefficients du problème sont au format binaire (QUBO/HUBO) ou spin. Les deux possibilités sont `"spin"` ou `"binary"`                                                   | Oui      | N/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | Nom du backend pour effectuer la requête                                                                                                                                                  | Oui       | N/A                                                      | `"ibm_fez"`                                                                                                          |                                                                                         |
| options      | `Dict[str, Any]`   | Options pour gérer la soumission matérielle, comme le nombre de shots. Pour plus de détails sur la configuration des options, voir la section [Options](#options) | Non       | Pour voir les valeurs par défaut de la configuration des options, consulter la section [Options](#options)                                                         | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |
#### Formats de problème acceptés
Les arguments `problem` et `problem_type` encodent un problème d'optimisation de la forme

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
où

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- En choisissant `problem_type = "binary"`, tu indiques que la fonction de coût est au format `binary`, ce qui signifie que $D = {0,  1}^{n}$, c'est-à-dire que la fonction de coût est écrite dans la formulation QUBO/HUBO.
- En revanche, en choisissant `problem_type = "spin"`, la fonction de coût est écrite dans la formulation d'Ising, où $D = {-1, 1}^{n}$.

Les coefficients du problème doivent être encodés dans un dictionnaire comme suit :
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Note que les clés du dictionnaire doivent être des chaînes contenant un tuple valide d'entiers non répétés.
#### Options
Iskay offre des capacités de réglage fin grâce à des paramètres optionnels. Bien que les valeurs par défaut fonctionnent bien pour la plupart des problèmes, tu peux personnaliser le comportement pour des besoins spécifiques :
| Paramètre | Type | Défaut | Description |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | Mesures quantiques par itération (plus élevé = plus précis) |
| **num_iterations** | `int` | 10 | Itérations de l'algorithme (plus d'itérations peuvent améliorer la qualité de la solution) |
| **use_session** | `bool` | True | Utilise les sessions IBM pour réduire les temps d'attente en file |
| **seed_transpiler** | `int` | None | Défini pour une compilation reproductible du circuit quantique |
| **direct_qubit_mapping** | `bool` | False | Mappe les qubits virtuels directement aux qubits physiques |
| **job_tags** | `List[str]` | None | Tags personnalisés pour le suivi des tâches |
| **preprocessing_level** | `int` | 0 | Intensité du prétraitement du problème (0-3) - voir les détails ci-dessous |
| **postprocessing_level** | `int` | 2 | Niveau de raffinement de la solution (0-2) - voir les détails ci-dessous |
| **transpilation_level** | `int` | 0 | Essais d'optimisation du Transpiler (0-5) - voir les détails ci-dessous |
| **transpile_only** | `bool` | False | Analyse l'optimisation du circuit sans exécuter l'algorithme complet |

**Niveaux de prétraitement (0-3)** : Particulièrement important pour les problèmes plus importants qui ne peuvent pas actuellement tenir dans les temps de cohérence du matériel. Des niveaux de prétraitement plus élevés permettent d'atteindre des profondeurs de circuit plus faibles grâce à des approximations lors de la transpilation du problème :
- **Niveau 0** : Exact, circuits plus longs
- **Niveau 1** : Bon équilibre entre précision et approximation, en éliminant uniquement les portes dont les angles se situent dans le 10e centile le plus bas
- **Niveau 2** : Approximation légèrement plus élevée, en éliminant les portes dont les angles se situent dans le 20e centile le plus bas et en utilisant `approximation_degree=0.95` lors de la transpilation
- **Niveau 3** : Niveau d'approximation maximal, en éliminant les portes dans le 30e centile le plus bas et en utilisant `approximation_degree=0.90` lors de la transpilation

**Niveaux de transpilation (0-5)** : Contrôle les essais avancés d'optimisation du Transpiler pour la compilation des circuits quantiques. Cela peut entraîner une augmentation de la surcharge classique et, dans certains cas, ne pas modifier la profondeur du circuit. La valeur par défaut `2` conduit en général au circuit le plus petit et est relativement rapide.
- **Niveau 0** : Optimisation du circuit DCQO décomposé (disposition, routage, ordonnancement)
- **Niveau 1** : Optimisation de `PauliEvolutionGate`, puis du circuit DCQO décomposé (max_trials=10)
- **Niveau 2** : Optimisation de `PauliEvolutionGate`, puis du circuit DCQO décomposé (max_trials=15)
- **Niveau 3** : Optimisation de `PauliEvolutionGate`, puis du circuit DCQO décomposé (max_trials=20)
- **Niveau 4** : Optimisation de `PauliEvolutionGate`, puis du circuit DCQO décomposé (max_trials=25)
- **Niveau 5** : Optimisation de `PauliEvolutionGate`, puis du circuit DCQO décomposé (max_trials=50)

**Niveaux de post-traitement (0-2)** : Contrôle la quantité d'optimisation classique, compensant les erreurs de flip de bit avec différents nombres de passes greedy d'une recherche locale :
- **Niveau 0** : 1 passe
- **Niveau 1** : 2 passes
- **Niveau 2** : 3 passes

**Mode transpile uniquement** : Désormais disponible pour les utilisateurs qui souhaitent analyser l'optimisation du circuit sans exécuter l'algorithme quantique complet.

**Exemple de configuration personnalisée** : Voici comment tu pourrais configurer Iskay avec différents paramètres :

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Optimisation du seed** : Note que `seed_transpiler` est défini à `None` par défaut. Cela permet le processus d'optimisation automatique du Transpiler. Quand il vaut `None`, le système lance un essai avec plusieurs seeds et sélectionne celui qui produit la meilleure profondeur de circuit, tirant parti de la puissance totale du paramètre `max_trials` pour chaque niveau de transpilation.

**Performance des niveaux de transpilation** : Augmenter le nombre de `max_trials` avec des valeurs plus élevées pour `transpilation_level` augmentera inévitablement le temps de transpilation, mais cela ne modifiera pas toujours le circuit final — cela dépend fortement de la structure et de la complexité spécifiques du circuit. Pour certains circuits/problèmes, cependant, la différence entre 10 essais (niveau 1) et 50 essais (niveau 5) peut être dramatique, donc explorer ces paramètres pourrait être la clé pour trouver avec succès une solution.
### Sortie
| Nom   | Type               | Description                                                           | Exemple                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Solution et métadonnées. La structure varie en fonction de l'option `transpile_only`. |   Voir « Contenu du dictionnaire de résultat » ci-dessous      |
### Contenu du dictionnaire de résultat
La structure du dictionnaire de résultat dépend du mode d'exécution :

| Champ         | Type               | Mode | Description                                                                                                             | Exemple |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **Standard** | La solution mappée triée où les clés sont les indices de variables (sous forme de chaînes) triés numériquement et les valeurs sont les valeurs de variable correspondantes (1/-1 pour les problèmes spin, 1/0 pour les problèmes binaires).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Standard** | Informations détaillées sur la solution (voir les détails ci-dessous)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Standard** | Le type de problème d'optimisation ('spin' ou 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Transpile uniquement** | Analyse du circuit et détails de transpilation (voir les détails ci-dessous) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Exécution standard
Quand le paramètre optionnel `transpile_only=False` :

**Dictionnaire solution_info :**
- **"bitstring"** (`str`) : La représentation brute en bitstring de la solution.
- **"cost"** (`float`) : La valeur de coût/énergie associée à la solution.
- **"seed_transpiler"** (`int`) : La graine aléatoire utilisée pour le Transpiler qui a produit ce résultat.
- **"mapping"** (`Dict[int, int]`) : Le mappage original qubit-vers-variable utilisé dans le calcul.
- **"qpu_time"** (`float`, optionnel) : Le temps d'exécution QPU en secondes.

**Notes sur le mappage des variables :**
- Le dictionnaire `solution` est obtenu à partir du bitstring de solution, en utilisant l'objet `mapping` pour indexer les variables.
- Quand `problem_type=spin`, on utilise l'assignation $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- Les clés du dictionnaire solution sont les indices de variables triés numériquement sous forme de chaînes.

#### Analyse de transpilation
Quand le paramètre optionnel `transpile_only=True` :

**Dictionnaire transpilation_info :**
- **"best_seed"** (`int`) : La graine optimale trouvée pour la transpilation
- **"transpilation_time_seconds"** (`float`) : Temps pris pour le processus de transpilation
- **"transpiled_circuit"** (`Dict`) : Analyse du circuit contenant :
  - **"depth"** (`int`) : Profondeur du circuit (nombre de couches)
  - **"gate_count"** (`int`) : Nombre total de portes dans le circuit
  - **"num_qubits"** (`int`) : Nombre de qubits utilisés
  - **"width"** (`int`) : Largeur du circuit
  - **"operations"** (`Dict[str, int]`) : Décompte de chaque type de porte utilisé

**Utilisation du mode transpile uniquement :**
- Disponible pour les utilisateurs qui souhaitent analyser l'optimisation du circuit sans exécuter l'algorithme quantique complet.
- Utile pour l'analyse des circuits, les études d'optimisation de profondeur et la compréhension des effets de la transpilation avant de s'engager dans une exécution complète.
## Premiers pas
Dans cette documentation, nous allons parcourir les étapes d'utilisation de l'Iskay Quantum Optimizer. Au cours du processus, nous montrerons rapidement comment charger la fonction depuis le catalogue et comment convertir ton problème en une entrée valide, tout en montrant comment tu peux expérimenter différents paramètres optionnels.

Pour un exemple plus détaillé, consulte le tutoriel [Résoudre le problème de Market Split avec l'Iskay Quantum Optimizer de Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), où nous parcourons l'ensemble du processus d'utilisation d'Iskay Solver pour aborder le problème de Market Split, qui représente un défi d'allocation de ressources réel où les marchés doivent être partitionnés en régions de vente équilibrées pour répondre à des objectifs de demande précis.

Authentifie-toi en utilisant ta clé API, disponible sur le [tableau de bord IBM Quantum Platform](http://quantum.cloud.ibm.com/), et sélectionne la fonction Qiskit comme suit :

In [ ]:
# ruff: noqa: F821

> **Note:** Le code suivant suppose que tu as sauvegardé tes identifiants. Si ce n'est pas le cas, suis les instructions dans [sauvegarder ton compte IBM Cloud](/guides/functions#install-qiskit-functions-catalog-client) pour t'authentifier avec ta clé API.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Exemple 1 : Fonction de coût simple

Considère la fonction de coût dans la formulation spin :
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

où $(x_0, ..., x_4) \in {-1, 1}^5$.

La solution de cette fonction de coût simple est
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
avec la valeur minimale $C^{*} = -6$

### 1. Créer la fonction objectif

On commence par créer un dictionnaire avec les coefficients de la fonction objectif comme suit :

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Exécuter l'Optimizer
On résout le problème en exécutant l'optimizer. Puisque $(x_0, ..., x_4) \in {-1, 1}^5$, on doit définir `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Récupérer le résultat
La solution du problème d'optimisation est fournie directement par l'optimizer.

In [ ]:
print(job.result())

Cela affichera un dictionnaire de la forme :

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Remarque que le dictionnaire `solution` affiche le vecteur résultat $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Exemple 2 : MaxCut

De nombreux problèmes de graphes comme MaxCut ou l'ensemble indépendant maximum sont des problèmes NP-difficiles et des candidats idéaux pour tester les algorithmes quantiques et le matériel. Cet exemple montre comment résoudre le problème MaxCut d'un graphe 3-régulier avec le Quantum Optimizer.

Pour exécuter cet exemple, tu dois installer le paquet `networkx` en plus de `qiskit-ibm-catalog`. Pour l'installer, exécute la commande suivante :

In [ ]:
# %pip install networkx numpy

### 1. Créer la fonction objectif
Commence par générer un graphe 3-régulier aléatoire. Pour ce graphe, on définit la fonction objectif du problème MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Exécuter l'Optimizer
Résous le problème en exécutant l'optimizer.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Récupérer le résultat
Récupère le résultat et mappe le bitstring de solution en retour vers les nœuds du graphe d'origine.

In [ ]:
print(job.result())

La solution du problème Maxcut est directement contenue dans le sous-dictionnaire `solution` de l'objet résultat

In [ ]:
maxcut_solution = job.result()["solution"]

## Exemple 3 : Instances de benchmark
Les instances de benchmark sont disponibles sur GitHub : [Kipu benchmark instances.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

Les instances peuvent être chargées à l'aide de la bibliothèque `pygithub`. Pour l'installer, exécute la commande suivante :

In [ ]:
# %pip install pygithub

Les chemins des instances de benchmark sont :

**Maxcut :**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO :**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Pour reproduire la performance du benchmark pour les instances HUBO, sélectionne le backend `ibm_marrakesh` et définis `direct_qubit_mapping` à `True` dans le sous-dictionnaire `options`.
L'exemple suivant exécute l'instance Maxcut avec 32 nœuds.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Cas d'utilisation
Les cas d'utilisation typiques du solveur d'optimisation sont les problèmes d'optimisation combinatoire. Tu peux résoudre des problèmes issus de nombreux secteurs comme la finance, la pharmacie ou la logistique. Quelques exemples suivent.
* Optimisation de portefeuille (QUBO) : [publication scientifique](https://doi.org/10.1103/PhysRevApplied.22.054037) et [livre blanc](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Repliement de protéines (HUBO) : [publication scientifique](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Planification logistique (QUBO) : [publication scientifique](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Optimisation de réseaux : [webinaire](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market split (QUBO) : [tutoriel](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Si tu es intéressé par l'étude d'un cas d'utilisation spécifique et le développement d'un mappage dédié, nous pouvons t'aider. [Contacte-nous.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
## Obtenir de l'aide
Pour du soutien, contacte support@kipu-quantum.com.
## Prochaines étapes
- [Demander l'accès au Quantum Optimizer par Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Essaie le tutoriel [Résoudre le problème de Market Split avec l'Iskay Quantum Optimizer de Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Consulte [Romero, S. V., et al. (2025). Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Consulte [Cadavid, A. G., et al. (2024). Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Consulte [Chandarana, P., et al. (2025). Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)
## Informations supplémentaires
Iskay, tout comme le nom de notre entreprise Kipu Quantum, est un mot péruvien. Bien que nous soyons une startup allemande, ces mots viennent du pays natal de l'un de nos co-fondateurs, où le Quipu fut l'une des toutes premières machines à calculer développées par l'humanité, il y a 2000 ans avant J.-C.